# Recurrent Neural Networks (RNNs) and Long Short-Term Memory (LSTM) Networks
## A Comprehensive Deep Dive into Sequential Learning

---

**Table of Contents**
1. Introduction to Sequential Data
2. Vanilla RNN — Architecture & Mathematics
3. Forward Propagation in RNNs
4. Backpropagation Through Time (BPTT)
5. The Vanishing & Exploding Gradient Problem
6. Long Short-Term Memory (LSTM) Networks
7. LSTM Gates — Detailed Mathematics
8. LSTM Backpropagation
9. Gated Recurrent Unit (GRU) — A Simplified Alternative
10. Working Code Implementations
11. Industry Applications
12. Summary & References

---

## 1. Introduction to Sequential Data

### Why Standard Neural Networks Fail on Sequences

Traditional feedforward neural networks assume that all inputs are **independent** of each other. However, many real-world problems involve **sequential data** where the order matters:

- **Natural Language**: "The cat sat on the mat" — each word depends on prior context
- **Time Series**: Stock prices, sensor readings, weather data
- **Audio/Speech**: Temporal waveform patterns
- **Video**: Sequences of frames with temporal dependencies
- **DNA/Protein Sequences**: Biological sequence patterns

### The Core Problem

Given a sequence $$x = (x_1, x_2, \ldots, x_T)$$, we want to model the conditional probability:

$$P(y_t | x_1, x_2, \ldots, x_t)$$

This requires a **memory mechanism** — the network must remember relevant information from previous time steps. Feedforward networks have no such mechanism because:

1. They accept **fixed-size** inputs
2. They have **no parameter sharing** across different positions in the sequence
3. They have **no memory** of previous computations

### The Solution: Recurrence

The key idea is to introduce a **hidden state** $$h_t$$ that acts as a compressed summary of all past information:

$$h_t = f(h_{t-1}, x_t)$$

This recurrence relation is the foundation of all RNN architectures.

## 2. Vanilla RNN — Architecture & Mathematics

### Architecture Overview

A Recurrent Neural Network processes sequences by maintaining a **hidden state** that is updated at each time step. The same set of parameters is shared across all time steps (weight sharing).

### Network Components

| Component | Symbol | Dimensions | Description |
| --- | --- | --- | --- |
| Input at time $$t$$ | $$x_t$$ | $$\mathbb{R}^{d}$$ | Input vector (e.g., word embedding) |
| Hidden state at time $$t$$ | $$h_t$$ | $$\mathbb{R}^{n}$$ | Memory/hidden representation |
| Output at time $$t$$ | $$y_t$$ | $$\mathbb{R}^{m}$$ | Network output |
| Input-to-hidden weights | $$W_{xh}$$ | $$\mathbb{R}^{n \times d}$$ | Transforms input to hidden space |
| Hidden-to-hidden weights | $$W_{hh}$$ | $$\mathbb{R}^{n \times n}$$ | Transforms previous hidden state |
| Hidden-to-output weights | $$W_{hy}$$ | $$\mathbb{R}^{m \times n}$$ | Transforms hidden to output space |
| Hidden bias | $$b_h$$ | $$\mathbb{R}^{n}$$ | Bias for hidden computation |
| Output bias | $$b_y$$ | $$\mathbb{R}^{m}$$ | Bias for output computation |

### The RNN Equations

**Step 1: Compute the new hidden state**

$$h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$$

**Step 2: Compute the output**

$$y_t = W_{hy} \cdot h_t + b_y$$

**Step 3: Apply output activation** (task-dependent)

- Classification: $$\hat{y}_t = \text{softmax}(y_t)$$
- Regression: $$\hat{y}_t = y_t$$ (identity)

### Why $$\tanh$$?

The hyperbolic tangent is preferred because:
- Output range $$[-1, 1]$$ — allows the hidden state to represent both positive and negative activations
- Zero-centered — gradients don't suffer from a consistent bias in one direction
- Stronger gradients than sigmoid near zero (derivative peaks at 1.0 vs 0.25)

$$\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$$

$$\frac{d}{dz}\tanh(z) = 1 - \tanh^2(z)$$

## 3. Forward Propagation in RNNs

### Unrolling the RNN Through Time

When we "unroll" an RNN for a sequence of length $$T$$, we get a deep network with $$T$$ layers, each sharing the same weights:

```
x_1 → [RNN Cell] → h_1 → y_1
                ↓
x_2 → [RNN Cell] → h_2 → y_2
                ↓
x_3 → [RNN Cell] → h_3 → y_3
                ↓
              ...
                ↓
x_T → [RNN Cell] → h_T → y_T
```

### Complete Forward Pass Algorithm

**Initialize:** $$h_0 = \mathbf{0}$$ (or learned initialization)

**For** $$t = 1, 2, \ldots, T$$:

$$\quad a_t = W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h \quad \text{(pre-activation)}$$

$$\quad h_t = \tanh(a_t) \quad \text{(hidden state)}$$

$$\quad o_t = W_{hy} \cdot h_t + b_y \quad \text{(output logits)}$$

$$\quad \hat{y}_t = \text{softmax}(o_t) \quad \text{(prediction)}$$

### Loss Function

For sequence classification/generation, we use **cross-entropy loss** summed over time steps:

$$\mathcal{L} = -\sum_{t=1}^{T} \sum_{k=1}^{m} y_{t,k} \log(\hat{y}_{t,k})$$

Or in compact form:

$$\mathcal{L} = \sum_{t=1}^{T} \mathcal{L}_t$$

where $$\mathcal{L}_t = -\log(\hat{y}_{t, c_t})$$ is the loss at time $$t$$ for the correct class $$c_t$$.

### Types of RNN Architectures

| Architecture | Input → Output | Example |
| --- | --- | --- |
| One-to-One | Fixed input → Fixed output | Standard feedforward |
| One-to-Many | Single input → Sequence output | Image captioning |
| Many-to-One | Sequence input → Single output | Sentiment analysis |
| Many-to-Many (synced) | Sequence → Sequence (same length) | POS tagging |
| Many-to-Many (encoder-decoder) | Sequence → Sequence (different length) | Machine translation |

In [0]:
import numpy as np

class VanillaRNN:
    """
    Vanilla RNN implementation from scratch using NumPy.
    Demonstrates the forward and backward pass mechanics.
    """
    
    def __init__(self, input_size, hidden_size, output_size):
        """
        Initialize RNN parameters with Xavier initialization.
        
        Args:
            input_size (int): Dimension of input vectors (d)
            hidden_size (int): Dimension of hidden state (n)
            output_size (int): Dimension of output vectors (m)
        """
        # Xavier initialization for better gradient flow
        scale_xh = np.sqrt(2.0 / (input_size + hidden_size))
        scale_hh = np.sqrt(2.0 / (hidden_size + hidden_size))
        scale_hy = np.sqrt(2.0 / (hidden_size + output_size))
        
        # Weight matrices
        self.W_xh = np.random.randn(hidden_size, input_size) * scale_xh   # (n, d)
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale_hh  # (n, n)
        self.W_hy = np.random.randn(output_size, hidden_size) * scale_hy  # (m, n)
        
        # Bias vectors
        self.b_h = np.zeros((hidden_size, 1))   # (n, 1)
        self.b_y = np.zeros((output_size, 1))   # (m, 1)
        
        self.hidden_size = hidden_size
    
    def forward(self, inputs, h_prev=None):
        """
        Forward pass through the RNN for a full sequence.
        
        Args:
            inputs: List of input vectors, each of shape (d, 1)
            h_prev: Initial hidden state (n, 1). Defaults to zeros.
            
        Returns:
            outputs: List of output vectors
            hiddens: List of hidden states (for backprop)
        """
        if h_prev is None:
            h_prev = np.zeros((self.hidden_size, 1))
        
        hiddens = {-1: h_prev}  # Store h_{-1} = h_0 for BPTT
        outputs = []
        
        for t, x_t in enumerate(inputs):
            # Core RNN computation: h_t = tanh(W_xh * x_t + W_hh * h_{t-1} + b_h)
            a_t = self.W_xh @ x_t + self.W_hh @ hiddens[t-1] + self.b_h
            h_t = np.tanh(a_t)
            
            # Output: y_t = W_hy * h_t + b_y
            o_t = self.W_hy @ h_t + self.b_y
            y_t = self._softmax(o_t)
            
            hiddens[t] = h_t
            outputs.append(y_t)
        
        return outputs, hiddens
    
    @staticmethod
    def _softmax(x):
        """Numerically stable softmax."""
        e_x = np.exp(x - np.max(x))
        return e_x / e_x.sum()

# --- Demonstration ---
np.random.seed(42)

# Parameters
input_size = 10    # e.g., vocabulary size or embedding dim
hidden_size = 32   # hidden state dimension
output_size = 10   # output classes
seq_length = 5     # sequence length

# Create RNN
rnn = VanillaRNN(input_size, hidden_size, output_size)

# Generate random input sequence (one-hot encoded)
inputs = [np.random.randn(input_size, 1) for _ in range(seq_length)]

# Forward pass
outputs, hiddens = rnn.forward(inputs)

print("=" * 60)
print("VANILLA RNN FORWARD PASS DEMONSTRATION")
print("=" * 60)
print(f"\nInput size (d):    {input_size}")
print(f"Hidden size (n):   {hidden_size}")
print(f"Output size (m):   {output_size}")
print(f"Sequence length:   {seq_length}")
print(f"\nWeight shapes:")
print(f"  W_xh: {rnn.W_xh.shape} (hidden x input)")
print(f"  W_hh: {rnn.W_hh.shape} (hidden x hidden)")
print(f"  W_hy: {rnn.W_hy.shape} (output x hidden)")
print(f"\nFor each time step t:")
for t in range(seq_length):
    print(f"  t={t}: h_t shape={hiddens[t].shape}, "
          f"h_t range=[{hiddens[t].min():.3f}, {hiddens[t].max():.3f}], "
          f"output sum={outputs[t].sum():.4f} (should be ~1.0)")

## 4. Backpropagation Through Time (BPTT)

### The Challenge

Since the RNN shares parameters across time steps, computing gradients requires propagating errors **backward through all time steps**. This is called **Backpropagation Through Time (BPTT)**.

### Deriving the Gradients

The total loss is:

$$\mathcal{L} = \sum_{t=1}^{T} \mathcal{L}_t$$

#### Gradient w.r.t. Output Weights $$W_{hy}$$

Since $$o_t = W_{hy} \cdot h_t + b_y$$ and $$\hat{y}_t = \text{softmax}(o_t)$$:

$$\frac{\partial \mathcal{L}}{\partial W_{hy}} = \sum_{t=1}^{T} \frac{\partial \mathcal{L}_t}{\partial o_t} \cdot h_t^\top$$

where $$\frac{\partial \mathcal{L}_t}{\partial o_t} = \hat{y}_t - y_t$$ (for softmax + cross-entropy).

#### Gradient w.r.t. Hidden State $$h_t$$

The hidden state $$h_t$$ influences the loss at time $$t$$ **and all future time steps** through $$h_{t+1}, h_{t+2}, \ldots, h_T$$:

$$\frac{\partial \mathcal{L}}{\partial h_t} = \frac{\partial \mathcal{L}_t}{\partial h_t} + \frac{\partial \mathcal{L}}{\partial h_{t+1}} \cdot \frac{\partial h_{t+1}}{\partial h_t}$$

The first term (direct contribution to output):

$$\frac{\partial \mathcal{L}_t}{\partial h_t} = W_{hy}^\top \cdot (\hat{y}_t - y_t)$$

The Jacobian of $$h_{t+1}$$ w.r.t. $$h_t$$:

$$\frac{\partial h_{t+1}}{\partial h_t} = \text{diag}(1 - h_{t+1}^2) \cdot W_{hh}$$

where $$\text{diag}(1 - h_{t+1}^2)$$ is the derivative of $$\tanh$$.

#### Gradient w.r.t. $$W_{hh}$$ (The Critical Gradient)

$$\frac{\partial \mathcal{L}}{\partial W_{hh}} = \sum_{t=1}^{T} \sum_{k=1}^{t} \frac{\partial \mathcal{L}_t}{\partial h_t} \left(\prod_{j=k+1}^{t} \frac{\partial h_j}{\partial h_{j-1}}\right) \frac{\partial h_k}{\partial W_{hh}}$$

The key term is the **product of Jacobians**:

$$\prod_{j=k+1}^{t} \frac{\partial h_j}{\partial h_{j-1}} = \prod_{j=k+1}^{t} \text{diag}(1 - h_j^2) \cdot W_{hh}$$

This product is the source of the **vanishing/exploding gradient problem**.

### BPTT Algorithm (Pseudocode)

```
Initialize: dW_xh = 0, dW_hh = 0, dW_hy = 0, db_h = 0, db_y = 0
Initialize: dh_next = 0

For t = T, T-1, ..., 1:
    # Output gradient
    do_t = y_hat_t - y_t
    dW_hy += do_t @ h_t.T
    db_y += do_t
    
    # Hidden state gradient (from output + from future)
    dh_t = W_hy.T @ do_t + dh_next
    
    # Through tanh
    da_t = dh_t * (1 - h_t**2)
    
    # Parameter gradients
    dW_xh += da_t @ x_t.T
    dW_hh += da_t @ h_{t-1}.T
    db_h += da_t
    
    # Propagate to previous time step
    dh_next = W_hh.T @ da_t
```

In [0]:
class VanillaRNN_WithBackprop(VanillaRNN):
    """
    Extended RNN with full BPTT implementation.
    """
    
    def backward(self, inputs, targets, outputs, hiddens):
        """
        Backpropagation Through Time (BPTT).
        
        Args:
            inputs: List of input vectors (d, 1)
            targets: List of target indices (integers)
            outputs: List of softmax outputs from forward pass
            hiddens: Dict of hidden states from forward pass
            
        Returns:
            gradients: Dict of parameter gradients
            loss: Total cross-entropy loss
        """
        T = len(inputs)
        
        # Initialize gradients
        dW_xh = np.zeros_like(self.W_xh)
        dW_hh = np.zeros_like(self.W_hh)
        dW_hy = np.zeros_like(self.W_hy)
        db_h = np.zeros_like(self.b_h)
        db_y = np.zeros_like(self.b_y)
        
        dh_next = np.zeros_like(hiddens[0])  # gradient flowing from future
        loss = 0.0
        
        # Backward pass through time
        for t in reversed(range(T)):
            # Cross-entropy loss at time t
            loss += -np.log(outputs[t][targets[t], 0] + 1e-12)
            
            # Gradient of loss w.r.t. output (softmax + cross-entropy)
            do_t = outputs[t].copy()
            do_t[targets[t]] -= 1  # d(L)/d(o_t) = y_hat - y
            
            # Gradients for W_hy and b_y
            dW_hy += do_t @ hiddens[t].T
            db_y += do_t
            
            # Gradient w.r.t. hidden state h_t
            # Comes from two sources: (1) current output, (2) next time step
            dh_t = self.W_hy.T @ do_t + dh_next
            
            # Backprop through tanh: d/dz tanh(z) = 1 - tanh²(z)
            da_t = dh_t * (1 - hiddens[t] ** 2)
            
            # Accumulate parameter gradients
            dW_xh += da_t @ inputs[t].T
            dW_hh += da_t @ hiddens[t-1].T
            db_h += da_t
            
            # Gradient to pass to previous time step
            dh_next = self.W_hh.T @ da_t
        
        # Gradient clipping to prevent exploding gradients
        for grad in [dW_xh, dW_hh, dW_hy, db_h, db_y]:
            np.clip(grad, -5, 5, out=grad)
        
        gradients = {
            'W_xh': dW_xh, 'W_hh': dW_hh, 'W_hy': dW_hy,
            'b_h': db_h, 'b_y': db_y
        }
        
        return gradients, loss
    
    def train_step(self, inputs, targets, learning_rate=0.01):
        """
        Single training step: forward + backward + update.
        """
        # Forward pass
        outputs, hiddens = self.forward(inputs)
        
        # Backward pass
        gradients, loss = self.backward(inputs, targets, outputs, hiddens)
        
        # Parameter update (SGD)
        self.W_xh -= learning_rate * gradients['W_xh']
        self.W_hh -= learning_rate * gradients['W_hh']
        self.W_hy -= learning_rate * gradients['W_hy']
        self.b_h -= learning_rate * gradients['b_h']
        self.b_y -= learning_rate * gradients['b_y']
        
        return loss

# --- Train on a simple character-level sequence ---
np.random.seed(42)

# Simple character vocabulary
chars = list("hello world")
vocab = sorted(set(chars))
char_to_idx = {ch: i for i, ch in enumerate(vocab)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(vocab)

print(f"Vocabulary: {vocab}")
print(f"Vocab size: {vocab_size}")
print(f"Char to index: {char_to_idx}")

# Create training data: predict next character
def one_hot(idx, size):
    v = np.zeros((size, 1))
    v[idx] = 1.0
    return v

# Prepare training sequence
input_chars = chars[:-1]   # 'hello worl'
target_chars = chars[1:]   # 'ello world'

inputs = [one_hot(char_to_idx[ch], vocab_size) for ch in input_chars]
targets = [char_to_idx[ch] for ch in target_chars]

# Train
rnn_bp = VanillaRNN_WithBackprop(vocab_size, hidden_size=64, output_size=vocab_size)

print("\n" + "=" * 60)
print("TRAINING: Character-level RNN (predicting next character)")
print("=" * 60)
print(f"Sequence: '{''.join(input_chars)}' -> '{''.join(target_chars)}'")
print(f"\nTraining progress:")

losses = []
for epoch in range(201):
    loss = rnn_bp.train_step(inputs, targets, learning_rate=0.01)
    losses.append(loss)
    if epoch % 50 == 0:
        print(f"  Epoch {epoch:4d}: Loss = {loss:.4f}")

print(f"\nFinal loss: {losses[-1]:.4f} (started at {losses[0]:.4f})")
print(f"Loss reduction: {((losses[0] - losses[-1]) / losses[0] * 100):.1f}%")

## 5. The Vanishing & Exploding Gradient Problem

### Mathematical Foundation

Recall the gradient of the loss w.r.t. $$W_{hh}$$ involves products of Jacobians:

$$\prod_{j=k+1}^{t} \frac{\partial h_j}{\partial h_{j-1}} = \prod_{j=k+1}^{t} \text{diag}(1 - h_j^2) \cdot W_{hh}$$

Let's denote $$D_j = \text{diag}(1 - h_j^2)$$ (diagonal matrix of tanh derivatives). Then:

$$\prod_{j=k+1}^{t} D_j \cdot W_{hh}$$

### Analysis via Singular Values

If we consider the spectral norm (largest singular value) $$\sigma_{\max}$$ of $$W_{hh}$$:

$$\left\| \prod_{j=k+1}^{t} D_j \cdot W_{hh} \right\| \leq \prod_{j=k+1}^{t} \|D_j\| \cdot \|W_{hh}\|$$

Since $$\tanh'(z) \in (0, 1]$$, we have $$\|D_j\| \leq 1$$. Therefore:

$$\left\| \prod_{j=k+1}^{t} D_j \cdot W_{hh} \right\| \leq \|W_{hh}\|^{t-k}$$

### Two Regimes

**Case 1: $$\sigma_{\max}(W_{hh}) < 1$$ (Vanishing Gradients)**

$$\left\| \prod_{j=k+1}^{t} D_j \cdot W_{hh} \right\| \to 0 \quad \text{exponentially as } (t-k) \to \infty$$

The network **cannot learn long-range dependencies** because gradients from distant time steps are negligibly small.

**Case 2: $$\sigma_{\max}(W_{hh}) > 1$$ (Exploding Gradients)**

$$\left\| \prod_{j=k+1}^{t} D_j \cdot W_{hh} \right\| \to \infty \quad \text{exponentially}$$

Gradients grow uncontrollably, causing numerical instability (NaN values, oscillating loss).

### Practical Consequences

| Problem | Symptom | Mitigation |
| --- | --- | --- |
| Vanishing gradients | Loss plateaus, no long-term learning | LSTM/GRU, residual connections |
| Exploding gradients | NaN loss, erratic updates | Gradient clipping, weight initialization |

### Gradient Clipping

The standard mitigation for exploding gradients:

$$\hat{g} = \begin{cases} g & \text{if } \|g\| \leq \theta \\ \frac{\theta}{\|g\|} \cdot g & \text{if } \|g\| > \theta \end{cases}$$

where $$\theta$$ is the clipping threshold (typically 1.0 or 5.0).

In [0]:
import matplotlib.pyplot as plt

def demonstrate_vanishing_gradient(hidden_size=100, seq_lengths=[10, 50, 100, 200]):
    """
    Demonstrate how gradient magnitude decays with sequence length.
    Shows the product of Jacobians ||D_j * W_hh||^(t-k) over time.
    """
    np.random.seed(42)
    
    results = {}
    
    for seq_len in seq_lengths:
        # Initialize W_hh with spectral norm slightly less than 1
        W_hh = np.random.randn(hidden_size, hidden_size) * 0.5 / np.sqrt(hidden_size)
        
        # Track gradient norm over time steps
        gradient_norms = []
        
        # Simulate forward pass to get hidden states
        h = np.random.randn(hidden_size, 1) * 0.1
        
        # Compute cumulative Jacobian product
        jacobian_product = np.eye(hidden_size)
        
        for t in range(seq_len):
            # Simulate tanh activation
            h = np.tanh(W_hh @ h)
            
            # Diagonal matrix of tanh derivatives
            D_t = np.diag((1 - h.flatten() ** 2))
            
            # Accumulate Jacobian product
            jacobian_product = D_t @ W_hh @ jacobian_product
            
            # Track the norm
            norm = np.linalg.norm(jacobian_product)
            gradient_norms.append(norm)
        
        results[seq_len] = gradient_norms
    
    # Plotting
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Linear scale
    ax = axes[0]
    for seq_len, norms in results.items():
        ax.plot(norms, label=f'Seq len={seq_len}', alpha=0.8)
    ax.set_xlabel('Time Steps Back', fontsize=12)
    ax.set_ylabel('Gradient Norm (Jacobian Product)', fontsize=12)
    ax.set_title('Vanishing Gradient: Linear Scale', fontsize=13)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Log scale
    ax = axes[1]
    for seq_len, norms in results.items():
        log_norms = [np.log10(n + 1e-300) for n in norms]
        ax.plot(log_norms, label=f'Seq len={seq_len}', alpha=0.8)
    ax.set_xlabel('Time Steps Back', fontsize=12)
    ax.set_ylabel('log₁₀(Gradient Norm)', fontsize=12)
    ax.set_title('Vanishing Gradient: Log Scale', fontsize=13)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.axhline(y=-10, color='red', linestyle='--', alpha=0.5, label='Effectively zero')
    
    plt.tight_layout()
    plt.suptitle('Demonstration: Gradient Vanishes Exponentially with Depth', 
                 fontsize=14, y=1.02)
    plt.show()
    
    # Print analysis
    print("\n" + "=" * 60)
    print("VANISHING GRADIENT ANALYSIS")
    print("=" * 60)
    print(f"\nHidden size: {hidden_size}")
    print(f"Spectral norm of W_hh: {np.linalg.norm(W_hh, 2):.4f}")
    print(f"\nGradient norm at various depths:")
    for seq_len, norms in results.items():
        final_norm = norms[-1]
        print(f"  After {seq_len:3d} steps: {final_norm:.2e} "
              f"({'vanished' if final_norm < 1e-10 else 'still present'})")

demonstrate_vanishing_gradient()

## 6. Long Short-Term Memory (LSTM) Networks

### Historical Context

LSTM was introduced by **Hochreiter & Schmidhuber (1997)** specifically to address the vanishing gradient problem. The core insight is to create a **separate memory pathway** (the cell state) that allows gradients to flow unchanged over many time steps.

### Key Design Principles

1. **Cell State ($$C_t$$)**: A dedicated memory line that runs through the entire sequence with minimal interaction — like a conveyor belt
2. **Gating Mechanisms**: Learned gates that control what information to **add**, **remove**, or **output** from the cell state
3. **Additive Updates**: The cell state is updated via **addition** (not multiplication), preventing gradient vanishing

### LSTM vs Vanilla RNN

| Feature | Vanilla RNN | LSTM |
| --- | --- | --- |
| Memory | Single hidden state $$h_t$$ | Cell state $$C_t$$ + hidden state $$h_t$$ |
| Update | Multiplicative (tanh) | Additive (gated) |
| Gradient flow | Exponential decay | Near-constant (via cell state) |
| Parameters | $$O(n^2)$$ | $$O(4n^2)$$ (4x more) |
| Long-range memory | Poor (10-20 steps) | Excellent (100s of steps) |

### The Four Gates of LSTM

An LSTM cell contains four neural network layers (gates):

1. **Forget Gate ($$f_t$$)**: Decides what to discard from cell state
2. **Input Gate ($$i_t$$)**: Decides what new information to store
3. **Candidate Cell State ($$\tilde{C}_t$$)**: Creates new candidate values
4. **Output Gate ($$o_t$$)**: Decides what to output from cell state

Each gate is a sigmoid layer (output in $$[0, 1]$$) acting as a soft switch:
- $$0$$ = completely block this information
- $$1$$ = completely let this information through

## 7. LSTM Gates — Detailed Mathematics

### Notation

- $$x_t \in \mathbb{R}^d$$: input at time $$t$$
- $$h_{t-1} \in \mathbb{R}^n$$: previous hidden state
- $$C_{t-1} \in \mathbb{R}^n$$: previous cell state
- $$\sigma$$: sigmoid function $$\sigma(z) = \frac{1}{1 + e^{-z}}$$
- $$[h_{t-1}, x_t]$$: concatenation of vectors

---

### Gate 1: The Forget Gate ($$f_t$$)

**Purpose**: Decide which elements of the old cell state $$C_{t-1}$$ to retain.

$$f_t = \sigma\left(W_f \cdot [h_{t-1}, x_t] + b_f\right)$$

where:
- $$W_f \in \mathbb{R}^{n \times (n+d)}$$ are the forget gate weights
- $$b_f \in \mathbb{R}^n$$ is the forget gate bias
- Output: $$f_t \in (0, 1)^n$$ — element-wise retention probability

**Intuition**: For language modeling, when a new subject appears, the forget gate learns to reset the gender/number features of the old subject.

**Bias initialization**: Typically initialized to 1.0 (biasing towards remembering) to help early training.

---

### Gate 2: The Input Gate ($$i_t$$)

**Purpose**: Decide which elements of the new candidate values to write into the cell state.

$$i_t = \sigma\left(W_i \cdot [h_{t-1}, x_t] + b_i\right)$$

where:
- $$W_i \in \mathbb{R}^{n \times (n+d)}$$ are the input gate weights
- $$b_i \in \mathbb{R}^n$$ is the input gate bias
- Output: $$i_t \in (0, 1)^n$$ — element-wise write probability

---

### Gate 3: Candidate Cell State ($$\tilde{C}_t$$)

**Purpose**: Create new candidate values that **could** be added to the cell state.

$$\tilde{C}_t = \tanh\left(W_C \cdot [h_{t-1}, x_t] + b_C\right)$$

where:
- $$W_C \in \mathbb{R}^{n \times (n+d)}$$ are the candidate weights
- $$b_C \in \mathbb{R}^n$$ is the candidate bias
- Output: $$\tilde{C}_t \in (-1, 1)^n$$ — proposed new memory values

**Why tanh here?** Allows both positive and negative updates, centering the cell state values.

---

### Cell State Update

**The core update equation** — this is where the magic happens:

$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

where $$\odot$$ denotes element-wise (Hadamard) product.

**This is an additive update!** The gradient can flow through $$C_t$$ back to $$C_{t-1}$$ via the $$f_t$$ path with minimal decay (if $$f_t \approx 1$$). This is the key difference from vanilla RNNs.

Compare:
- **RNN**: $$h_t = \tanh(W \cdot h_{t-1} + \ldots)$$ — always squashed through tanh
- **LSTM**: $$C_t = f_t \odot C_{t-1} + \ldots$$ — linear self-loop when $$f_t = 1$$

---

### Gate 4: The Output Gate ($$o_t$$)

**Purpose**: Decide what parts of the cell state to expose as the hidden state output.

$$o_t = \sigma\left(W_o \cdot [h_{t-1}, x_t] + b_o\right)$$

$$h_t = o_t \odot \tanh(C_t)$$

where:
- $$W_o \in \mathbb{R}^{n \times (n+d)}$$ are the output gate weights
- $$h_t$$ is the final hidden state output (also used for predictions)

**Intuition**: The cell state might store information about verb conjugation, but if the current token is a noun, the output gate suppresses verb-related features.

---

### Complete LSTM Equations Summary

$$\boxed{\begin{aligned}
f_t &= \sigma(W_f \cdot [h_{t-1}, x_t] + b_f) & \text{(forget gate)} \\
i_t &= \sigma(W_i \cdot [h_{t-1}, x_t] + b_i) & \text{(input gate)} \\
\tilde{C}_t &= \tanh(W_C \cdot [h_{t-1}, x_t] + b_C) & \text{(candidate)} \\
C_t &= f_t \odot C_{t-1} + i_t \odot \tilde{C}_t & \text{(cell update)} \\
o_t &= \sigma(W_o \cdot [h_{t-1}, x_t] + b_o) & \text{(output gate)} \\
h_t &= o_t \odot \tanh(C_t) & \text{(hidden state)}
\end{aligned}}$$

### Parameter Count

Total parameters for a single LSTM layer:
- 4 weight matrices: $$4 \times n \times (n + d)$$
- 4 bias vectors: $$4 \times n$$
- **Total**: $$4n(n + d) + 4n = 4n(n + d + 1)$$

For $$n = 512, d = 256$$: $$4 \times 512 \times (512 + 256 + 1) = 1,574,912$$ parameters per layer.

In [0]:
class LSTMCell:
    """
    Single LSTM cell implementation from scratch.
    Demonstrates the complete gating mechanism.
    """
    
    def __init__(self, input_size, hidden_size):
        """
        Initialize LSTM parameters.
        
        Args:
            input_size (int): Dimension of input (d)
            hidden_size (int): Dimension of hidden/cell state (n)
        """
        self.input_size = input_size
        self.hidden_size = hidden_size
        concat_size = hidden_size + input_size  # [h_{t-1}, x_t]
        
        # Xavier initialization
        scale = np.sqrt(2.0 / (concat_size + hidden_size))
        
        # Forget gate parameters
        self.W_f = np.random.randn(hidden_size, concat_size) * scale
        self.b_f = np.ones((hidden_size, 1))  # Initialize to 1 (remember by default)
        
        # Input gate parameters
        self.W_i = np.random.randn(hidden_size, concat_size) * scale
        self.b_i = np.zeros((hidden_size, 1))
        
        # Candidate cell state parameters
        self.W_c = np.random.randn(hidden_size, concat_size) * scale
        self.b_c = np.zeros((hidden_size, 1))
        
        # Output gate parameters
        self.W_o = np.random.randn(hidden_size, concat_size) * scale
        self.b_o = np.zeros((hidden_size, 1))
    
    @staticmethod
    def sigmoid(x):
        """Numerically stable sigmoid."""
        return np.where(x >= 0, 
                       1 / (1 + np.exp(-x)), 
                       np.exp(x) / (1 + np.exp(x)))
    
    def forward_step(self, x_t, h_prev, C_prev):
        """
        Single time step forward pass.
        
        Args:
            x_t: Input at time t, shape (d, 1)
            h_prev: Previous hidden state, shape (n, 1)
            C_prev: Previous cell state, shape (n, 1)
            
        Returns:
            h_t: New hidden state (n, 1)
            C_t: New cell state (n, 1)
            cache: Intermediate values for backprop
        """
        # Concatenate [h_{t-1}, x_t]
        concat = np.vstack([h_prev, x_t])  # (n+d, 1)
        
        # === FORGET GATE ===
        # f_t = sigmoid(W_f @ [h_{t-1}, x_t] + b_f)
        f_t = self.sigmoid(self.W_f @ concat + self.b_f)
        
        # === INPUT GATE ===
        # i_t = sigmoid(W_i @ [h_{t-1}, x_t] + b_i)
        i_t = self.sigmoid(self.W_i @ concat + self.b_i)
        
        # === CANDIDATE CELL STATE ===
        # C_tilde = tanh(W_c @ [h_{t-1}, x_t] + b_c)
        C_tilde = np.tanh(self.W_c @ concat + self.b_c)
        
        # === CELL STATE UPDATE ===
        # C_t = f_t * C_{t-1} + i_t * C_tilde
        C_t = f_t * C_prev + i_t * C_tilde
        
        # === OUTPUT GATE ===
        # o_t = sigmoid(W_o @ [h_{t-1}, x_t] + b_o)
        o_t = self.sigmoid(self.W_o @ concat + self.b_o)
        
        # === HIDDEN STATE ===
        # h_t = o_t * tanh(C_t)
        h_t = o_t * np.tanh(C_t)
        
        # Cache for backprop
        cache = {
            'concat': concat, 'f_t': f_t, 'i_t': i_t,
            'C_tilde': C_tilde, 'C_t': C_t, 'o_t': o_t,
            'h_t': h_t, 'C_prev': C_prev, 'h_prev': h_prev
        }
        
        return h_t, C_t, cache
    
    def forward(self, inputs):
        """
        Forward pass through entire sequence.
        
        Args:
            inputs: List of input vectors, each (d, 1)
            
        Returns:
            hiddens: List of hidden states
            cells: List of cell states
            caches: List of caches for backprop
        """
        T = len(inputs)
        h_t = np.zeros((self.hidden_size, 1))
        C_t = np.zeros((self.hidden_size, 1))
        
        hiddens, cells, caches = [], [], []
        
        for t in range(T):
            h_t, C_t, cache = self.forward_step(inputs[t], h_t, C_t)
            hiddens.append(h_t)
            cells.append(C_t)
            caches.append(cache)
        
        return hiddens, cells, caches


# --- Demonstrate LSTM Forward Pass ---
np.random.seed(42)

input_size = 10
hidden_size = 32
seq_length = 8

lstm = LSTMCell(input_size, hidden_size)
inputs = [np.random.randn(input_size, 1) for _ in range(seq_length)]

hiddens, cells, caches = lstm.forward(inputs)

print("=" * 60)
print("LSTM FORWARD PASS DEMONSTRATION")
print("=" * 60)
print(f"\nInput size: {input_size}, Hidden size: {hidden_size}, Seq length: {seq_length}")
print(f"\nParameter count: {4 * hidden_size * (hidden_size + input_size + 1):,}")
print(f"\nGate statistics at each time step:")
print(f"{'t':>3} | {'f_t mean':>10} | {'i_t mean':>10} | {'o_t mean':>10} | {'|C_t|':>10} | {'|h_t|':>10}")
print("-" * 70)

for t in range(seq_length):
    cache = caches[t]
    print(f"{t:3d} | {cache['f_t'].mean():10.4f} | {cache['i_t'].mean():10.4f} | "
          f"{cache['o_t'].mean():10.4f} | {np.abs(cells[t]).mean():10.4f} | "
          f"{np.abs(hiddens[t]).mean():10.4f}")

print(f"\nNote: Forget gate initialized ~0.73 (bias=1 through sigmoid)")
print(f"This ensures the LSTM remembers by default in early training.")

## 8. LSTM Backpropagation

### Why LSTM Solves the Vanishing Gradient

Consider the gradient of the loss w.r.t. the cell state at time $$k < t$$:

$$\frac{\partial C_t}{\partial C_k} = \prod_{j=k+1}^{t} \frac{\partial C_j}{\partial C_{j-1}} = \prod_{j=k+1}^{t} f_j$$

Since $$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$:

$$\frac{\partial C_t}{\partial C_{t-1}} = f_t \quad \text{(element-wise)}$$

**Key insight**: If the forget gate $$f_t \approx 1$$, the gradient flows **unchanged** through the cell state! Compare:

- **RNN gradient**: $$\prod_{j} \text{diag}(1-h_j^2) \cdot W_{hh}$$ — exponential decay
- **LSTM gradient**: $$\prod_{j} f_j$$ — can remain near 1 if forget gates are open

### Full LSTM Backward Pass

Define: $$\delta h_t = \frac{\partial \mathcal{L}}{\partial h_t}$$ and $$\delta C_t = \frac{\partial \mathcal{L}}{\partial C_t}$$

At each time step $$t$$ (going backwards):

**Step 1**: Gradient w.r.t. output gate
$$\delta o_t = \delta h_t \odot \tanh(C_t)$$
$$\delta \hat{o}_t = \delta o_t \odot o_t \odot (1 - o_t) \quad \text{(sigmoid derivative)}$$

**Step 2**: Gradient w.r.t. cell state (from output + from future)
$$\delta C_t = \delta C_t + \delta h_t \odot o_t \odot (1 - \tanh^2(C_t))$$

**Step 3**: Gradient w.r.t. forget gate
$$\delta f_t = \delta C_t \odot C_{t-1}$$
$$\delta \hat{f}_t = \delta f_t \odot f_t \odot (1 - f_t)$$

**Step 4**: Gradient w.r.t. input gate
$$\delta i_t = \delta C_t \odot \tilde{C}_t$$
$$\delta \hat{i}_t = \delta i_t \odot i_t \odot (1 - i_t)$$

**Step 5**: Gradient w.r.t. candidate
$$\delta \tilde{C}_t = \delta C_t \odot i_t$$
$$\delta \hat{C}_t = \delta \tilde{C}_t \odot (1 - \tilde{C}_t^2) \quad \text{(tanh derivative)}$$

**Step 6**: Concatenate all gate gradients
$$\delta z_t = [\delta \hat{f}_t; \delta \hat{i}_t; \delta \hat{C}_t; \delta \hat{o}_t]$$

**Step 7**: Parameter gradients
$$\delta W += \delta z_t \cdot [h_{t-1}, x_t]^\top$$
$$\delta b += \delta z_t$$

**Step 8**: Propagate to previous time step
$$\delta [h_{t-1}, x_t] = W^\top \cdot \delta z_t$$
$$\delta C_{t-1} = \delta C_t \odot f_t \quad \text{(gradient flows through forget gate!)}$$

### Gradient Flow Comparison

$$\text{RNN}: \quad \frac{\partial h_T}{\partial h_1} = \prod_{t=2}^{T} \text{diag}(1-h_t^2) \cdot W_{hh} \approx 0 \text{ for large } T$$

$$\text{LSTM}: \quad \frac{\partial C_T}{\partial C_1} = \prod_{t=2}^{T} f_t \approx 1 \text{ if } f_t \approx 1$$

This is analogous to **residual connections** in deep feedforward networks (ResNets), where $$y = x + F(x)$$ allows gradient to bypass layers.

## 9. Gated Recurrent Unit (GRU) — A Simplified Alternative

### Motivation

Introduced by **Cho et al. (2014)**, GRU simplifies LSTM by:
- Merging the cell state and hidden state into a single state $$h_t$$
- Combining the forget and input gates into an **update gate**
- Using a **reset gate** instead of separate output gate

### GRU Equations

$$\boxed{\begin{aligned}
z_t &= \sigma(W_z \cdot [h_{t-1}, x_t] + b_z) & \text{(update gate)} \\
r_t &= \sigma(W_r \cdot [h_{t-1}, x_t] + b_r) & \text{(reset gate)} \\
\tilde{h}_t &= \tanh(W_h \cdot [r_t \odot h_{t-1}, x_t] + b_h) & \text{(candidate)} \\
h_t &= (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t & \text{(final state)}
\end{aligned}}$$

### Gate Interpretation

**Update Gate ($$z_t$$)**: Balances between keeping old information and accepting new.
- $$z_t \to 0$$: Keep previous hidden state (copy mode)
- $$z_t \to 1$$: Fully replace with candidate

**Note**: $$h_t = (1-z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$ is a convex combination! The forget and input operations are **coupled** — unlike LSTM where they're independent.

**Reset Gate ($$r_t$$)**: Controls how much of the past to forget when computing the candidate.
- $$r_t \to 0$$: Ignore previous hidden state entirely (fresh start)
- $$r_t \to 1$$: Use full previous state

### LSTM vs GRU Comparison

| Feature | LSTM | GRU |
| --- | --- | --- |
| States | $$h_t$$ and $$C_t$$ | Only $$h_t$$ |
| Gates | 3 (forget, input, output) | 2 (update, reset) |
| Parameters | $$4n(n+d+1)$$ | $$3n(n+d+1)$$ |
| Forget/Input coupling | Independent | Coupled via $$z_t$$ |
| Performance | Better on long sequences | Often comparable, faster |
| Training speed | Slower (more params) | Faster (fewer params) |

### When to Use Which?

- **LSTM**: Default choice for long sequences, complex dependencies, or when you need maximum representational capacity
- **GRU**: Smaller datasets, faster training needed, shorter sequences, mobile/edge deployment
- **Empirically**: Performance is often similar; GRU wins on compute efficiency

In [0]:
def compare_gradient_flow(seq_length=100, hidden_size=64, num_trials=50):
    """
    Compare gradient flow in RNN, LSTM, and GRU over long sequences.
    Demonstrates why LSTM/GRU solve the vanishing gradient problem.
    """
    np.random.seed(42)
    
    rnn_norms = []
    lstm_norms = []
    gru_norms = []
    
    for trial in range(num_trials):
        # --- Vanilla RNN gradient product ---
        W_hh = np.random.randn(hidden_size, hidden_size) * 0.5 / np.sqrt(hidden_size)
        rnn_grad = np.eye(hidden_size)
        h = np.random.randn(hidden_size, 1) * 0.1
        
        for t in range(seq_length):
            h = np.tanh(W_hh @ h)
            D = np.diag((1 - h.flatten()**2))
            rnn_grad = D @ W_hh @ rnn_grad
        rnn_norms.append(np.linalg.norm(rnn_grad))
        
        # --- LSTM gradient product (through cell state) ---
        # Gradient = product of forget gates
        lstm_grad = 1.0
        for t in range(seq_length):
            # Simulate forget gate (sigmoid of random input, biased towards 1)
            f_t = 1 / (1 + np.exp(-(np.random.randn() * 0.5 + 1.0)))  # biased ~0.7
            lstm_grad *= f_t
        lstm_norms.append(lstm_grad)
        
        # --- GRU gradient product ---
        # Gradient through (1 - z_t) path
        gru_grad = 1.0
        for t in range(seq_length):
            z_t = 1 / (1 + np.exp(-(np.random.randn() * 0.5)))  # ~0.5
            gru_grad *= (1 - z_t)
        gru_norms.append(gru_grad)
    
    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].hist(np.log10(np.array(rnn_norms) + 1e-300), bins=30, 
                 color='red', alpha=0.7, edgecolor='black')
    axes[0].set_title(f'Vanilla RNN\nGradient Norms (log₁₀)', fontsize=12)
    axes[0].set_xlabel('log₁₀(gradient norm)')
    axes[0].axvline(x=0, color='green', linestyle='--', label='No decay')
    axes[0].legend()
    
    axes[1].hist(np.log10(np.array(lstm_norms) + 1e-300), bins=30, 
                 color='blue', alpha=0.7, edgecolor='black')
    axes[1].set_title(f'LSTM (Cell State Path)\nGradient Norms (log₁₀)', fontsize=12)
    axes[1].set_xlabel('log₁₀(gradient norm)')
    axes[1].axvline(x=0, color='green', linestyle='--', label='No decay')
    axes[1].legend()
    
    axes[2].hist(np.log10(np.array(gru_norms) + 1e-300), bins=30,
                 color='purple', alpha=0.7, edgecolor='black')
    axes[2].set_title(f'GRU (Update Gate Path)\nGradient Norms (log₁₀)', fontsize=12)
    axes[2].set_xlabel('log₁₀(gradient norm)')
    axes[2].axvline(x=0, color='green', linestyle='--', label='No decay')
    axes[2].legend()
    
    plt.suptitle(f'Gradient Flow Comparison (Sequence Length = {seq_length})', 
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\n" + "=" * 60)
    print(f"GRADIENT FLOW COMPARISON (Sequence Length = {seq_length})")
    print("=" * 60)
    print(f"\n{'Architecture':<12} | {'Mean Gradient':<15} | {'Min':<12} | {'Max':<12} | {'Vanished?'}")
    print("-" * 75)
    
    rnn_mean = np.mean(rnn_norms)
    lstm_mean = np.mean(lstm_norms)
    gru_mean = np.mean(gru_norms)
    
    print(f"{'RNN':<12} | {rnn_mean:<15.2e} | {min(rnn_norms):<12.2e} | {max(rnn_norms):<12.2e} | {'YES' if rnn_mean < 1e-10 else 'Partial'}")
    print(f"{'LSTM':<12} | {lstm_mean:<15.2e} | {min(lstm_norms):<12.2e} | {max(lstm_norms):<12.2e} | {'YES' if lstm_mean < 1e-10 else 'NO'}")
    print(f"{'GRU':<12} | {gru_mean:<15.2e} | {min(gru_norms):<12.2e} | {max(gru_norms):<12.2e} | {'YES' if gru_mean < 1e-10 else 'NO'}")
    
    print(f"\n→ LSTM preserves ~{lstm_mean/rnn_mean:.0e}x more gradient than RNN")
    print(f"→ GRU preserves ~{gru_mean/rnn_mean:.0e}x more gradient than RNN")

compare_gradient_flow(seq_length=100)

## 10. Working Code Implementations

Below we implement complete, runnable examples using **PyTorch** — the industry-standard framework for RNN/LSTM development.

### 10.1 Sine Wave Prediction with LSTM

A classic demonstration: train an LSTM to predict future values of a sine wave given past observations. This illustrates:
- Sequence-to-sequence learning
- Time series forecasting
- How LSTM captures periodic patterns

In [0]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# =============================================
# LSTM Model for Time Series Prediction
# =============================================

class LSTMPredictor(nn.Module):
    """
    LSTM-based sequence predictor.
    Architecture: Input -> LSTM layers -> Fully Connected -> Output
    """
    
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1):
        super(LSTMPredictor, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM layer(s)
        # input_size: number of features per time step
        # hidden_size: number of LSTM units
        # num_layers: stacked LSTM layers
        # batch_first: input shape is (batch, seq_len, features)
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.1  # Dropout between LSTM layers
        )
        
        # Fully connected output layer
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x, hidden=None):
        """
        Forward pass.
        
        Args:
            x: Input tensor of shape (batch_size, seq_len, input_size)
            hidden: Optional (h_0, c_0) tuple
            
        Returns:
            output: Predictions of shape (batch_size, output_size)
            hidden: Final (h_n, c_n) state
        """
        # LSTM forward: output contains hidden states for all time steps
        # hidden = (h_n, c_n) - final states
        lstm_out, hidden = self.lstm(x, hidden)
        
        # Use only the last time step's hidden state for prediction
        # lstm_out shape: (batch, seq_len, hidden_size)
        last_hidden = lstm_out[:, -1, :]  # (batch, hidden_size)
        
        # Map to output
        output = self.fc(last_hidden)  # (batch, output_size)
        
        return output, hidden

# =============================================
# Data Preparation
# =============================================

def create_sine_data(seq_length=50, num_samples=1000, pred_steps=1):
    """
    Create sine wave sequences for training.
    Input: seq_length consecutive sine values
    Target: the next pred_steps values
    """
    # Generate a long sine wave with some complexity
    t = np.linspace(0, 100 * np.pi, num_samples + seq_length + pred_steps)
    data = np.sin(t) + 0.3 * np.sin(2.5 * t)  # Composite signal
    
    X, y = [], []
    for i in range(num_samples):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length:i + seq_length + pred_steps])
    
    X = np.array(X).reshape(-1, seq_length, 1).astype(np.float32)
    y = np.array(y).reshape(-1, pred_steps).astype(np.float32)
    
    return torch.tensor(X), torch.tensor(y)

# Create dataset
seq_length = 50
X_train, y_train = create_sine_data(seq_length=seq_length, num_samples=800)
X_test, y_test = create_sine_data(seq_length=seq_length, num_samples=200)

print("=" * 60)
print("LSTM SINE WAVE PREDICTION")
print("=" * 60)
print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Sequence length: {seq_length}")
print(f"Input shape: {X_train.shape} (batch, seq_len, features)")
print(f"Target shape: {y_train.shape} (batch, prediction_steps)")

# =============================================
# Training
# =============================================

model = LSTMPredictor(input_size=1, hidden_size=64, num_layers=2, output_size=1)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Print model architecture
print(f"\nModel Architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Training loop
num_epochs = 50
batch_size = 32
train_losses = []

print(f"\nTraining ({num_epochs} epochs, batch_size={batch_size}):")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    num_batches = 0
    
    # Mini-batch training
    indices = torch.randperm(X_train.shape[0])
    for i in range(0, X_train.shape[0], batch_size):
        batch_idx = indices[i:i + batch_size]
        X_batch = X_train[batch_idx]
        y_batch = y_train[batch_idx]
        
        # Forward pass
        predictions, _ = model(X_batch)
        loss = criterion(predictions, y_batch)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping (important for RNNs!)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    train_losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d}/{num_epochs}: Loss = {avg_loss:.6f}")

print(f"\nFinal training loss: {train_losses[-1]:.6f}")

In [0]:
# =============================================
# Evaluation & Visualization
# =============================================

model.eval()
with torch.no_grad():
    test_predictions, _ = model(X_test)
    test_loss = criterion(test_predictions, y_test)
    print(f"Test MSE Loss: {test_loss.item():.6f}")

# Visualize predictions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Training loss curve
ax = axes[0, 0]
ax.plot(train_losses, 'b-', linewidth=2)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('MSE Loss', fontsize=11)
ax.set_title('Training Loss Curve', fontsize=13)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# Plot 2: Predictions vs Actual (sample sequences)
ax = axes[0, 1]
sample_idx = list(range(0, 100))
actual = y_test[sample_idx].numpy().flatten()
predicted = test_predictions[sample_idx].numpy().flatten()

ax.plot(actual, 'b-', label='Actual', linewidth=1.5, alpha=0.8)
ax.plot(predicted, 'r--', label='Predicted', linewidth=1.5, alpha=0.8)
ax.set_xlabel('Sample Index', fontsize=11)
ax.set_ylabel('Value', fontsize=11)
ax.set_title('LSTM Predictions vs Actual', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 3: Scatter plot of predictions
ax = axes[1, 0]
ax.scatter(actual, predicted, alpha=0.5, s=20, c='blue')
ax.plot([actual.min(), actual.max()], [actual.min(), actual.max()], 
        'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Value', fontsize=11)
ax.set_ylabel('Predicted Value', fontsize=11)
ax.set_title('Prediction Accuracy Scatter', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Multi-step prediction (autoregressive)
ax = axes[1, 1]

# Use first test sequence and predict many steps ahead
seed_seq = X_test[0:1].clone()  # (1, seq_len, 1)
autoregressive_preds = []

for step in range(100):
    with torch.no_grad():
        pred, _ = model(seed_seq)
    autoregressive_preds.append(pred.item())
    # Shift window: remove first, append prediction
    new_input = pred.reshape(1, 1, 1)
    seed_seq = torch.cat([seed_seq[:, 1:, :], new_input], dim=1)

ax.plot(range(seq_length), X_test[0, :, 0].numpy(), 'b-', 
        linewidth=2, label='Input sequence')
ax.plot(range(seq_length, seq_length + 100), autoregressive_preds, 'r-', 
        linewidth=2, label='Autoregressive predictions')
ax.axvline(x=seq_length, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Time Step', fontsize=11)
ax.set_ylabel('Value', fontsize=11)
ax.set_title('Multi-Step Autoregressive Prediction', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Correlation coefficient
from scipy import stats
r, p = stats.pearsonr(actual, predicted)
print(f"\nPearson correlation: r = {r:.4f} (p = {p:.2e})")
print(f"Mean Absolute Error: {np.mean(np.abs(actual - predicted)):.4f}")

### 10.2 Character-Level Text Generation with LSTM

This example demonstrates:
- Character-level language modeling
- Temperature-controlled sampling
- How LSTMs learn grammatical structure from raw text

In [0]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =============================================
# Character-Level LSTM Language Model
# =============================================

class CharLSTM(nn.Module):
    """
    Character-level LSTM for text generation.
    
    Architecture:
        Embedding -> LSTM (stacked) -> Linear -> Softmax
    """
    
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=2):
        super(CharLSTM, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Character embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_size)
        
        # Stacked LSTM
        self.lstm = nn.LSTM(
            input_size=embed_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )
        
        # Output projection
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, hidden=None):
        """
        Args:
            x: Character indices, shape (batch, seq_len)
            hidden: (h_0, c_0) tuple
        """
        # Embed characters
        embeds = self.embedding(x)  # (batch, seq_len, embed_size)
        
        # LSTM forward
        lstm_out, hidden = self.lstm(embeds, hidden)
        # lstm_out: (batch, seq_len, hidden_size)
        
        # Project to vocabulary
        output = self.fc(lstm_out)  # (batch, seq_len, vocab_size)
        
        return output, hidden
    
    def generate(self, start_char_idx, char_to_idx, idx_to_char, 
                 length=200, temperature=0.8):
        """
        Generate text character by character.
        
        Temperature controls randomness:
            - temperature < 1: more conservative (higher confidence picks)
            - temperature = 1: standard sampling
            - temperature > 1: more creative/random
        """
        self.eval()
        generated = [start_char_idx]
        hidden = None
        
        with torch.no_grad():
            for _ in range(length):
                # Prepare input
                x = torch.tensor([[generated[-1]]])
                
                # Forward pass
                output, hidden = self(x, hidden)
                
                # Apply temperature scaling
                logits = output[0, -1, :] / temperature
                
                # Sample from distribution
                probs = F.softmax(logits, dim=0)
                next_char = torch.multinomial(probs, 1).item()
                
                generated.append(next_char)
        
        return ''.join([idx_to_char[idx] for idx in generated])


# =============================================
# Training on Sample Text
# =============================================

# Sample training text (simplified for demonstration)
text = """The quick brown fox jumps over the lazy dog. 
A recurrent neural network processes sequences step by step. 
Long short term memory networks solve the vanishing gradient problem. 
Deep learning has revolutionized natural language processing. 
Neural networks learn representations from data automatically. 
The hidden state carries information across time steps. 
Gradient clipping prevents exploding gradients during training. 
Attention mechanisms allow models to focus on relevant parts. 
Transformers have largely replaced recurrent models for NLP tasks. 
However LSTMs remain powerful for many sequential problems."""

# Build vocabulary
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

# Encode text
encoded = [char_to_idx[ch] for ch in text]

# Create training sequences
def create_char_sequences(encoded, seq_length=40):
    X, y = [], []
    for i in range(0, len(encoded) - seq_length):
        X.append(encoded[i:i + seq_length])
        y.append(encoded[i + 1:i + seq_length + 1])
    return torch.tensor(X), torch.tensor(y)

seq_length = 40
X_text, y_text = create_char_sequences(encoded, seq_length)

print("=" * 60)
print("CHARACTER-LEVEL LSTM TEXT GENERATOR")
print("=" * 60)
print(f"\nVocabulary size: {vocab_size} characters")
print(f"Characters: {''.join(chars)}")
print(f"Text length: {len(text)} chars")
print(f"Training sequences: {X_text.shape[0]}")
print(f"Sequence length: {seq_length}")

# Create model
char_model = CharLSTM(
    vocab_size=vocab_size,
    embed_size=32,
    hidden_size=128,
    num_layers=2
)

print(f"\nModel:")
print(f"  Parameters: {sum(p.numel() for p in char_model.parameters()):,}")

# Training
optimizer = torch.optim.Adam(char_model.parameters(), lr=0.003)
criterion = nn.CrossEntropyLoss()

char_model.train()
for epoch in range(101):
    # Random batch
    idx = torch.randint(0, X_text.shape[0], (64,))
    X_batch = X_text[idx]
    y_batch = y_text[idx]
    
    output, _ = char_model(X_batch)
    # Reshape for cross-entropy: (batch*seq_len, vocab_size) vs (batch*seq_len)
    loss = criterion(output.reshape(-1, vocab_size), y_batch.reshape(-1))
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(char_model.parameters(), 1.0)
    optimizer.step()
    
    if epoch % 25 == 0:
        print(f"  Epoch {epoch:3d}: Loss = {loss.item():.4f}")

# Generate text with different temperatures
print("\n" + "=" * 60)
print("TEXT GENERATION (after limited training)")
print("=" * 60)

for temp in [0.5, 0.8, 1.2]:
    start_idx = char_to_idx['T']
    generated = char_model.generate(start_idx, char_to_idx, idx_to_char, 
                                    length=120, temperature=temp)
    print(f"\n  Temperature={temp}: {generated[:100]}...")

### 10.3 Sentiment Analysis with Bidirectional LSTM

This example demonstrates:
- **Bidirectional LSTMs**: Process sequence in both directions for richer context
- **Text classification** (many-to-one architecture)
- **Attention-weighted pooling** for interpretable predictions

**Bidirectional LSTM Math:**

The forward LSTM processes $$x_1, x_2, \ldots, x_T$$ left-to-right:
$$\overrightarrow{h_t} = \text{LSTM}_{\text{forward}}(x_t, \overrightarrow{h_{t-1}})$$

The backward LSTM processes $$x_T, x_{T-1}, \ldots, x_1$$ right-to-left:
$$\overleftarrow{h_t} = \text{LSTM}_{\text{backward}}(x_t, \overleftarrow{h_{t+1}})$$

Final representation: $$h_t = [\overrightarrow{h_t}; \overleftarrow{h_t}]$$ (concatenation, dimension $$2n$$)

In [0]:
import torch
import torch.nn as nn

# =============================================
# Bidirectional LSTM for Sentiment Analysis
# =============================================

class BiLSTMSentiment(nn.Module):
    """
    Bidirectional LSTM for binary sentiment classification.
    Architecture: Embedding -> BiLSTM -> Attention Pooling -> FC -> Sigmoid
    """
    
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=2, 
                 dropout=0.3, pad_idx=0):
        super(BiLSTMSentiment, self).__init__()
        
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=pad_idx)
        
        self.lstm = nn.LSTM(
            input_size=embed_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        
        # Self-attention for weighted pooling
        self.attention = nn.Linear(hidden_size * 2, 1)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )
    
    def attention_pool(self, lstm_output):
        """Attention-weighted sum: alpha_t = softmax(W @ h_t + b)"""
        scores = self.attention(lstm_output).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_output).squeeze(1)
        return context, weights
    
    def forward(self, x):
        embeds = self.embedding(x)
        lstm_out, (h_n, c_n) = self.lstm(embeds)
        context, attn_weights = self.attention_pool(lstm_out)
        logits = self.classifier(context)
        return logits, attn_weights

# --- Demonstration ---
torch.manual_seed(42)

sentiment_model = BiLSTMSentiment(
    vocab_size=1000, embed_size=64, hidden_size=128, num_layers=2, dropout=0.3
)

# Synthetic batch
X_batch = torch.randint(1, 1000, (16, 30))
y_batch = torch.randint(0, 2, (16, 1)).float()

logits, attn_weights = sentiment_model(X_batch)

print("=" * 60)
print("BIDIRECTIONAL LSTM SENTIMENT CLASSIFIER")
print("=" * 60)
print(f"\nModel Architecture:")
print(sentiment_model)
print(f"\nTotal parameters: {sum(p.numel() for p in sentiment_model.parameters()):,}")
print(f"\nForward pass:")
print(f"  Input: {X_batch.shape} -> Output logits: {logits.shape}")
print(f"  Attention weights: {attn_weights.shape} (per-token importance)")
print(f"  Predictions: {torch.sigmoid(logits[:5]).flatten().detach().numpy().round(3)}")
print(f"  Attention sum = {attn_weights[0].sum().item():.4f} (should be 1.0)")

# Quick training
optimizer = torch.optim.Adam(sentiment_model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

sentiment_model.train()
print(f"\nTraining demonstration:")
for epoch in range(6):
    logits, _ = sentiment_model(X_batch)
    loss = criterion(logits, y_batch)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(sentiment_model.parameters(), 1.0)
    optimizer.step()
    acc = ((torch.sigmoid(logits) > 0.5).float() == y_batch).float().mean()
    print(f"  Epoch {epoch}: Loss={loss.item():.4f}, Accuracy={acc.item():.2%}")

## 11. Industry Applications of RNNs and LSTMs

Despite the rise of Transformers, RNNs and LSTMs remain widely deployed across industries due to their efficiency on streaming data, lower memory requirements, and proven effectiveness on many sequential tasks.

---

### 11.1 Natural Language Processing (NLP)

| Application | Architecture | Companies/Products | Notes |
| --- | --- | --- | --- |
| Machine Translation | Seq2Seq LSTM + Attention | Google (pre-2017), Baidu | Encoder-decoder with attention mechanism |
| Text Summarization | BiLSTM + Copy Mechanism | Salesforce, Microsoft | Abstractive and extractive methods |
| Named Entity Recognition | BiLSTM-CRF | spaCy, Clinical NLP | CRF layer models label dependencies |
| Sentiment Analysis | BiLSTM + Attention | Amazon, Yelp, Twitter | Product reviews, social media |
| Chatbots/Dialogue | LSTM-based Seq2Seq | Early versions of Alexa, Siri | Response generation |
| Autocomplete/Suggestions | Character/Word LSTM | SwiftKey, Gboard | Next-word prediction on mobile |
| Spell Correction | Encoder-Decoder LSTM | Grammarly, Google Search | Sequence-to-sequence correction |

**Key Math for Seq2Seq with Attention:**

$$\alpha_{t,s} = \frac{\exp(\text{score}(h_t^{\text{dec}}, h_s^{\text{enc}}))}{\sum_{s'=1}^{S} \exp(\text{score}(h_t^{\text{dec}}, h_{s'}^{\text{enc}}))}$$

$$c_t = \sum_{s=1}^{S} \alpha_{t,s} \cdot h_s^{\text{enc}}$$

where $$\text{score}(h_t, h_s) = h_t^\top W_a h_s$$ (Luong attention)

---

### 11.2 Finance & Trading

| Application | Architecture | Impact |
| --- | --- | --- |
| Stock Price Prediction | Stacked LSTM | Capture temporal patterns, volatility clustering |
| Algorithmic Trading | LSTM + RL | Signal generation from price sequences |
| Fraud Detection | LSTM Autoencoder | Detect anomalous transaction sequences |
| Credit Risk Modeling | GRU on payment history | Sequential behavior patterns |
| Portfolio Optimization | LSTM + Attention | Dynamic asset allocation |

**Why LSTM works for finance:**
- Markets have **memory** — past prices influence future prices
- Volatility exhibits **clustering** (GARCH-like patterns)
- LSTM can model **regime changes** via forget gate

---

### 11.3 Healthcare & Biomedical

| Application | Architecture | Data Type |
| --- | --- | --- |
| Electronic Health Records | BiLSTM + Attention | Patient visit sequences |
| Drug Discovery | LSTM on SMILES strings | Molecular sequences |
| ECG Classification | 1D-CNN + LSTM | Time-series signals |
| ICU Mortality Prediction | LSTM on vital signs | Real-time monitoring |
| Protein Structure | BiLSTM (early methods) | Amino acid sequences |
| Clinical Note Analysis | LSTM-CRF | Unstructured medical text |

---

### 11.4 Speech & Audio

| Application | Architecture | Companies |
| --- | --- | --- |
| Speech Recognition (ASR) | Deep BiLSTM + CTC | Google (DeepSpeech), Baidu |
| Text-to-Speech (TTS) | LSTM + WaveNet | Google Tacotron (v1) |
| Music Generation | Stacked LSTM | Magenta (Google), AIVA |
| Speaker Verification | LSTM d-vectors | Google, Apple |
| Audio Event Detection | CRNN (CNN + RNN) | Smart home devices |

**CTC Loss for Speech:**
Connectionist Temporal Classification allows alignment-free training:
$$P(y|x) = \sum_{\pi \in \mathcal{B}^{-1}(y)} \prod_{t=1}^{T} P(\pi_t | x)$$
where $$\mathcal{B}^{-1}(y)$$ is the set of all valid alignments mapping to label $$y$$.

---

### 11.5 Autonomous Systems & Robotics

| Application | Architecture | Use Case |
| --- | --- | --- |
| Trajectory Prediction | Social LSTM | Pedestrian/vehicle path prediction |
| Robot Control | LSTM Policy Networks | Sequential decision making |
| Anomaly Detection | LSTM Autoencoders | Manufacturing sensor data |
| Autonomous Driving | CNN + LSTM | Temporal scene understanding |

---

### 11.6 E-Commerce & Recommendation

| Application | Architecture | Companies |
| --- | --- | --- |
| Session-based Recommendations | GRU4Rec | Spotify, Netflix (early) |
| Click-stream Analysis | LSTM on user sessions | Amazon, Flipkart |
| Dynamic Pricing | LSTM on demand patterns | Airlines, Uber |
| Search Query Understanding | BiLSTM | Google, Bing, Croma |
| Customer Churn Prediction | LSTM on engagement sequences | Telecom, SaaS |

---

### 11.7 Energy & IoT

| Application | Architecture | Scale |
| --- | --- | --- |
| Load Forecasting | Encoder-Decoder LSTM | National grid level |
| Predictive Maintenance | LSTM on sensor streams | Industrial equipment |
| Wind/Solar Forecasting | LSTM + Weather features | Renewable energy plants |
| Smart Meter Analytics | GRU on consumption | Millions of meters |

## 11.8 Advanced LSTM Variants Used in Industry

### Peephole Connections (Gers & Schmidhuber, 2000)

Allow gates to "peek" at the cell state directly:

$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + W_{pf} \cdot C_{t-1} + b_f)$$
$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + W_{pi} \cdot C_{t-1} + b_i)$$
$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + W_{po} \cdot C_t + b_o)$$

### Depth-Gated LSTM

Adds depth gating for stacked LSTMs:
$$h_t^{(l)} = d_t^{(l)} \odot h_t^{(l)} + (1 - d_t^{(l)}) \odot h_t^{(l-1)}$$

### Multiplicative LSTM (mLSTM, Krause et al., 2016)

Adds a multiplicative interaction:
$$m_t = (W_{mx} \cdot x_t) \odot (W_{mh} \cdot h_{t-1})$$

Used in character-level language modeling for better factorization of the input space.

### Quasi-RNN (Bradbury et al., 2016)

Hybrid CNN-RNN: uses convolutions for parallelizable computation and a minimal recurrent pooling step:
$$z_t = \tanh(W_z \cdot X_{t:t+k})$$ (convolution over input)
$$h_t = f_t \odot h_{t-1} + (1 - f_t) \odot z_t$$ (element-wise recurrence)

Achieves near-LSTM quality at 2-17x the speed.

---

### When Are RNNs/LSTMs Still Preferred Over Transformers?

| Scenario | Why LSTM/GRU | Why Not Transformer |
| --- | --- | --- |
| Real-time streaming | O(1) per step, constant memory | O(n²) attention, needs full sequence |
| Edge/mobile deployment | Smaller model size | Too many parameters |
| Very long sequences (>10k) | Linear time complexity | Quadratic attention cost |
| Online learning | Natural incremental updates | Requires recomputation |
| Limited training data | Fewer parameters, less overfitting | Data-hungry architecture |
| Causal generation | Natural left-to-right | Requires masking tricks |

In [0]:
# =============================================
# Practical Guide: LSTM Hyperparameters & Tips
# =============================================

print("=" * 70)
print("PRACTICAL GUIDE: LSTM HYPERPARAMETERS & BEST PRACTICES")
print("=" * 70)

guide = """
┌────────────────────────────────────────────────────────────────────┐
│                 HYPERPARAMETER RECOMMENDATIONS                      │
├─────────────────────┬────────────────────┬─────────────────────────┤
│ Hyperparameter        │ Typical Range        │ Notes                     │
├─────────────────────┼────────────────────┼─────────────────────────┤
│ Hidden size           │ 64 - 1024            │ 128-512 most common       │
│ Number of layers      │ 1 - 4               │ 2-3 usually sufficient    │
│ Learning rate         │ 1e-4 - 1e-2         │ Start with 1e-3 (Adam)    │
│ Gradient clip norm    │ 1.0 - 5.0           │ Essential for stability   │
│ Dropout               │ 0.1 - 0.5           │ Between LSTM layers       │
│ Batch size            │ 32 - 256            │ Smaller for longer seqs   │
│ Sequence length       │ 20 - 500            │ Task-dependent            │
│ Embedding dim         │ 50 - 300            │ Use pretrained if avail   │
│ Forget bias init      │ 1.0                 │ Bias towards remembering  │
└─────────────────────┴────────────────────┴─────────────────────────┘

┌────────────────────────────────────────────────────────────────────┐
│                    BEST PRACTICES CHECKLIST                         │
├────────────────────────────────────────────────────────────────────┤
│                                                                    │
│ ✓ Always use gradient clipping (prevents NaN loss)                 │
│ ✓ Initialize forget gate bias to 1.0                               │
│ ✓ Use bidirectional for classification tasks                       │
│ ✓ Apply dropout BETWEEN layers, not within recurrence              │
│ ✓ Sort sequences by length + use packed sequences (efficiency)     │
│ ✓ Use teacher forcing for seq2seq (with scheduled sampling)        │
│ ✓ Monitor gradient norms during training                           │
│ ✓ Use LayerNorm or BatchNorm between stacked layers                │
│ ✓ Consider learning rate warmup for very deep models               │
│ ✓ Truncated BPTT for very long sequences (>500 steps)              │
│                                                                    │
└────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────┐
│                     COMMON MISTAKES TO AVOID                        │
├────────────────────────────────────────────────────────────────────┤
│                                                                    │
│ ✗ Not detaching hidden states between batches (memory leak)        │
│ ✗ Using same hidden state init for all batches (should reset)      │
│ ✗ Forgetting to call model.eval() during inference                 │
│ ✗ Not normalizing/scaling input features                           │
│ ✗ Using too many LSTM layers (diminishing returns > 3)             │
│ ✗ Ignoring sequence length variation (use padding + masking)       │
│ ✗ Training without shuffling sequences between epochs              │
│                                                                    │
└────────────────────────────────────────────────────────────────────┘
"""
print(guide)

# Computational complexity comparison
print("\n" + "=" * 70)
print("COMPUTATIONAL COMPLEXITY COMPARISON")
print("=" * 70)
print(f"\n{'Model':<20} | {'Time Complexity':<20} | {'Space Complexity':<20} | {'Parallelizable'}")
print("-" * 85)
print(f"{'Vanilla RNN':<20} | {'O(T · n²)':<20} | {'O(n²)':<20} | {'No'}")
print(f"{'LSTM':<20} | {'O(T · 4n²)':<20} | {'O(4n²)':<20} | {'No'}")
print(f"{'GRU':<20} | {'O(T · 3n²)':<20} | {'O(3n²)':<20} | {'No'}")
print(f"{'Transformer':<20} | {'O(T² · d)':<20} | {'O(T² + d²)':<20} | {'Yes'}")
print(f"{'Quasi-RNN':<20} | {'O(T · kd)':<20} | {'O(kd²)':<20} | {'Partial'}")
print(f"\nwhere T=seq_length, n=hidden_size, d=model_dim, k=kernel_size")

## 12. Summary & References

### Key Takeaways

1. **RNNs** introduce recurrence to process sequential data, maintaining a hidden state as compressed memory
2. **Vanilla RNNs** suffer from vanishing/exploding gradients due to repeated multiplication of the $$W_{hh}$$ Jacobian
3. **LSTMs** solve this via an additive cell state update ($$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$) that acts as a gradient highway
4. **GRUs** offer a simpler alternative with fewer parameters, coupling forget and input via a single update gate
5. **Gradient clipping** is essential for training all recurrent architectures
6. **Bidirectional** processing captures both past and future context for classification tasks
7. **Despite Transformers**, RNNs/LSTMs remain relevant for streaming, edge deployment, and low-resource settings

### The Mathematical Essence

The entire story reduces to how gradients flow backward:

$$\text{Vanilla RNN}: \quad \frac{\partial h_T}{\partial h_1} = \prod_{t=2}^T \text{diag}(1-h_t^2) \cdot W_{hh} \xrightarrow{T \to \infty} 0 \text{ or } \infty$$

$$\text{LSTM}: \quad \frac{\partial C_T}{\partial C_1} = \prod_{t=2}^T f_t \approx 1 \text{ (when forget gates are open)}$$

---

### Foundational Papers

| Year | Paper | Contribution |
| --- | --- | --- |
| 1990 | Elman, "Finding Structure in Time" | Simple RNN architecture |
| 1994 | Bengio et al., "Learning Long-Term Dependencies..." | Formalized vanishing gradient problem |
| 1997 | Hochreiter & Schmidhuber, "Long Short-Term Memory" | Original LSTM |
| 2000 | Gers et al., "Learning to Forget" | Forget gate addition (modern LSTM) |
| 2000 | Gers et al., "Recurrent Nets that Time and Count" | Peephole connections |
| 2014 | Cho et al., "Learning Phrase Representations..." | GRU architecture |
| 2014 | Sutskever et al., "Sequence to Sequence Learning..." | Seq2Seq with LSTM |
| 2015 | Bahdanau et al., "Neural Machine Translation by..." | Attention mechanism |
| 2015 | Greff et al., "LSTM: A Search Space Odyssey" | Comprehensive LSTM variant comparison |
| 2017 | Vaswani et al., "Attention Is All You Need" | Transformer (LSTM successor for NLP) |

---

### Recommended Resources

- **Christopher Olah**: "Understanding LSTMs" (colah.github.io) — Best visual explanation
- **Andrej Karpathy**: "The Unreasonable Effectiveness of RNNs" — Practical intuition
- **Deep Learning Book** (Goodfellow et al.): Chapter 10 — Sequence Modeling
- **Stanford CS231n/CS224n**: Lecture notes on RNNs and LSTMs
- **PyTorch Documentation**: `torch.nn.LSTM`, `torch.nn.GRU` API reference

---

*This notebook provides a self-contained reference for understanding the theory, mathematics, implementation, and practical applications of Recurrent Neural Networks and LSTM architectures.*